# Green Route ML Pipeline
This notebook demonstrates dataset analysis, preprocessing, model training, evaluation and model selection for predicting CO2 emissions.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/routes_dataset.csv')
df.head()


## Dataset Overview

In [ ]:

df.info()
df.describe()
df.isnull().sum()


## Exploratory Data Analysis

In [ ]:

plt.figure(figsize=(6,4))
sns.histplot(df['distance_km'], bins=20)
plt.title('Distance Distribution')
plt.show()


In [ ]:

# simulate emission factors
emission_factors = {
    'bike':0.02,
    'car':0.12,
    'bus':0.05,
    'train':0.03,
    'flight':0.25
}

for mode,factor in emission_factors.items():
    df[mode+'_emission'] = df['distance_km']*factor

df.head()


In [ ]:

plt.figure(figsize=(8,4))
df[['bike_emission','car_emission','bus_emission','train_emission','flight_emission']].mean().plot(kind='bar')
plt.title("Average Emission per Mode")
plt.ylabel("CO2")
plt.show()


## Feature Engineering

In [ ]:

X = df[['distance_km']]
y = df['car_emission']  # target variable


## Train Test Split

In [ ]:

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)


## Model Training

In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models={
    "LinearRegression":LinearRegression(),
    "DecisionTree":DecisionTreeRegressor(),
    "RandomForest":RandomForestRegressor()
}

results={}

for name,model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    
    from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
    
    r2=r2_score(y_test,pred)
    mae=mean_absolute_error(y_test,pred)
    rmse=np.sqrt(mean_squared_error(y_test,pred))
    
    results[name]={"R2":r2,"MAE":mae,"RMSE":rmse}

results


## Model Comparison

In [ ]:

pd.DataFrame(results).T


## Select Best Model and Save

In [ ]:

best_model = RandomForestRegressor()
best_model.fit(X,y)

import joblib
joblib.dump(best_model,'../models/emission_model.pkl')
print("Model saved successfully")
